<font color=#009afa size=50>DAT5004-Big Data Management Systems</font>

<font size=50 color=#00c382>Task Three - MongoDB</font><br>
<font size=35>MongoDB Cluster and Titanic Dataset</font>

**Assessment Code:** PRAC1 (Task 3)

**Author:** Tom Rowan, ST20285213

**Course:** DAT5004_S2_24

**Course Tutor:** Dr. Sandeep Singh Sengar

<p>

## Install and Import Required Libraries

In [ ]:
%pip install -r requirements.txt

In [63]:
# Requests, CSV and base64
import requests
import csv
import io
import base64

# MongoDB
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

# Display
from pprint import pprint
from colorama import Fore, Style


## Define Utility Functions used to Display the results
Note: these functions were copied from the Cricketers Database (Task 2). The results table function has been modified to make it more generic so that it handles any list of dictionaries.

In [64]:
# Adds string padding to a string (left or right side)
def string_padding (a_string, length,side="right"):
    if len(a_string) < length:
        # Pad the string with spaces
        if side == "left":
            # Pad on the left
            return " " * (length - len(a_string)) + a_string        
        else:
            return a_string + " " * (length - len(a_string))    
    else:
        # Truncate the string to the specified length
        return a_string[:length]
    

# Print Results Table
def print_results_table (results, table_title = "Results", padding=16):

    # Set the padding value
    padding_value = padding

    if len(results) < 1:
        print (Fore.RED + "No results found" + Style.RESET_ALL)
    else:
        print ("")
        print (Fore.YELLOW + table_title + Style.RESET_ALL)

        header_fields = list(results[0].keys())

        header_row = ""
        for f in header_fields:
            #header_row = string_padding(f, len(str(results[0][f])),side='left')
            header_row += string_padding(f, padding_value,side='left')
        print ((len(header_row)+2) * "-")
        print (header_row)
        print ((len(header_row)+2) * "-")

        

        # Print each row of the results in alternating stripes
        row_count = 0
        for row in results:
            row_str = ""
            for f in header_fields:
                row_str += string_padding(str(row[f]), padding_value,side='left')
            if row_count % 2 == 0:
                print (Fore.CYAN + row_str + Style.RESET_ALL)
            else:   
                print (Fore.WHITE + row_str + Style.RESET_ALL)
            row_count += 1

        print ((len(header_row)+2) * "-")
        print ("Total Results: ", len(results))
        print ((len(header_row)+2) * "-")
        print ("")

# Task 3.1 - Create Cluster 

Here is a screenshot showing that I have created a cluster in MongoDB Atlas:

<img src='https://datascience.daemonchild.com/year2/dat5004/prac1/images/screenshot_cluster_creation.png' width=75%/>

# Task 3.2 - Loading Data

> Create a database and a collection within the current cluster of your MongoDB Atlas. You need to store the Titanic 
data in the collection. Using suitable Python codes, copy the data from Titanic DataFrame into the newly created 
collection within your database.

### Fetch the data from the Internet

The original Titanic dataset is located on the [Kaggle website](https://www.kaggle.com/datasets/yasserh/titanic-dataset). An account is required to obtain the data in a downloadable format. I have copied the CSV to my own website to allow easy programatic access without providing credentials. The dataset is otherwise unmodified.

It is located here: [https://datascience.daemonchild.com/year2/dat5004/prac1/datasets/Titanic-Dataset.csv](https://datascience.daemonchild.com/year2/dat5004/prac1/datasets/Titanic-Dataset.csv")

I have used the requests and csv libraries to import the data without without using pandas.

In [65]:
# Load Data from CSV
dataset_url = "https://datascience.daemonchild.com/year2/dat5004/prac1/datasets/Titanic-Dataset.csv"

# Fetch the CSV data
response = requests.get(url=dataset_url)
if response.status_code == 200:

    # Using io.StringIO to create a file object
    csv_string = io.StringIO(response.text)
    csv_data = csv.DictReader(csv_string) 

    # Convert to JSON 'Array'
    json_data = []  
    for row in csv_data: 
        # Add Dict to List
        json_data.append(row)
        
else:
    print(f"[FAILED] Failed to fetch data, got {response.status_code}")

# Print the first 3 rows of the data
print(f"[INFO] First 3 rows of the data, total {len(json_data)} rows")
pprint(json_data[0:3])

#  Create field_names variable for later use
field_names = list(json_data[0].keys())
field_names.sort()

[INFO] First 3 rows of the data, total 891 rows
[{'Age': '22',
  'Cabin': '',
  'Embarked': 'S',
  'Fare': '7.25',
  'Name': 'Braund, Mr. Owen Harris',
  'Parch': '0',
  'PassengerId': '1',
  'Pclass': '3',
  'Sex': 'male',
  'SibSp': '1',
  'Survived': '0',
  'Ticket': 'A/5 21171'},
 {'Age': '38',
  'Cabin': 'C85',
  'Embarked': 'C',
  'Fare': '71.2833',
  'Name': 'Cumings, Mrs. John Bradley (Florence Briggs Thayer)',
  'Parch': '0',
  'PassengerId': '2',
  'Pclass': '1',
  'Sex': 'female',
  'SibSp': '1',
  'Survived': '1',
  'Ticket': 'PC 17599'},
 {'Age': '26',
  'Cabin': '',
  'Embarked': 'S',
  'Fare': '7.925',
  'Name': 'Heikkinen, Miss. Laina',
  'Parch': '0',
  'PassengerId': '3',
  'Pclass': '3',
  'Sex': 'female',
  'SibSp': '0',
  'Survived': '1',
  'Ticket': 'STON/O2. 3101282'}]


### Connect to MongoDB

In [66]:
# Obfuscate the URI as it contains credentials
base64_encoded_uri = 'bW9uZ29kYitzcnY6Ly9kYXRhYmFzZS1hY2Nlc3M6bGV0LW1lLXVzZS10aGUtRDRUNEBjbHVzdGVyMC5wdWI0dm84Lm1vbmdvZGIubmV0Lz9yZXRyeVdyaXRlcz10cnVlJnc9bWFqb3JpdHkmYXBwTmFtZT1DbHVzdGVyMA=='
uri_bytes = base64.b64decode(base64_encoded_uri)
uri = uri_bytes.decode("ascii")

# Create a new client and connect to the server
client = MongoClient(uri, server_api=ServerApi('1'))

# Send a ping to confirm a successful connection
try:
    client.admin.command('ping')
    print("[OK] Successfully connected to MongoDB!")
except Exception as e:
    print("[FAIL] Failed to connect to MongoDB!")
    print(e)

[OK] Successfully connected to MongoDB!


### Upload the Data

In [67]:
database_name = "task3_database"
collection_name = "titanic_data"

if database_name in client.list_database_names():

    # NB: User cannot drop database due to permissions (it is NOT an admin, but can read/write from the database.)
    # client.drop_database(database_name)
    print(f"[INFO] Database exists: {database_name} - NOT uploading data!")

    print(f"[INFO] Connecting to database {database_name} and collection {collection_name}")
    task3_db = client["task3_database"]
    titanic_col = task3_db["titanic_data"]
    
else:

    print(f"[INFO] Creating database {database_name} with collection {collection_name}")
    # Create new database and collection
    task3_db = client["task3_database"]
    titanic_col = task3_db["titanic_data"]

    print (f"[INFO] Uploading {len(json_data)} rows")
    # Upload data to the MonogoDB collection
    _result_ = titanic_col.insert_many(json_data)


[INFO] Creating database task3_database with collection titanic_data
[INFO] Uploading 891 rows


Here we can see that the data has been uploaded to the cluster:

<img src='https://datascience.daemonchild.com/year2/dat5004/prac1/images/screenshot_titanic_data_in_cluster.png' width=75%/>

### Task: Add five additional documents

>Assume  that  the  original  creator  of  the  Titanic  data  added  five  extra  rows  with  incomplete  information  to  the 
Titanic_data.csv  file.  For  example,  the  new  rows  are  missing  details  such  as  ‘Name,’  ‘PassengerId,’  and 
‘Embarked.’ These inconsistencies could complicate data storage in a Relational Database, which requires strict 
adherence to schemas. However, MongoDB’s flexible schema allows you to handle this seamlessly. 

> Insert five additional documents into your existing collection, make sure that only the available fields are populated. Feel free 
to  use  dummy  data  to  supplement missing  information  and  make certain  your collection  remains  functional for 
analysis.

The following information regarding the meaning of the features was generated from information provided on the Kaggle webpage:

| Feature | Meaning |
|---|---|
|PassengerId | Passenger ID |
|Survived |Whether they Survived or not: 0 = No, 1 = Yes|
|Pclass|Ticket class: 1 = 1st, 2 = 2nd, 3 = 3rd|
|Name| Name of the Passenger|
|Sex |Gender (male or female)|
|Age|Age in Years|
|SibSp|No. of siblings / spouses aboard the Titanic|
|Parch|No. of parents / children aboard the Titanic|
|Ticket|Ticket number|
|Fare|Passenger fare|
|Embarked|Port of Embarkation:C = Cherbourg, Q = Queenstown, S = Southampton|

The next code block will add five additional documents to the collection. I have created dummy data based on a small family of two parents and three children. This allows me to populate the siblings and parents/children fields to maintain proper analysis. I have placed them in the same fictitional cabin.

In [68]:
_doc1 = {
  'Age': '40',
  'Cabin': 'ZZ9ZZA',
  'Embarked': 'S',
  'Fare': '42.00',
  'Name': 'Adams, Mr. Arthur',
  'Parch': '3',
  'PassengerId': 'ZZ9-001',
  'Pclass': '2',
  'Sex': 'male',
  'SibSp': '1',
  'Survived': '1',
  'Ticket': '9991'}

_doc2 = {
  'Age': '33',
  'Cabin': 'ZZ9ZZA',
  'Embarked': 'S',
  'Fare': '42.00',
  'Name': 'Adams, Mrs. Trillian',
  'Parch': '3',
  'PassengerId': 'ZZ9-002',
  'Pclass': '2',
  'Sex': 'female',
  'SibSp': '1',
  'Survived': '0',
  'Ticket': '9992'}

_doc3 = {
  'Age': '14',
  'Cabin': 'ZZ9ZZA',
  'Embarked': 'S',
  'Fare': '42.00',
  'Name': 'Adams, Master Zaphod',
  'Parch': '2',
  'PassengerId': 'ZZ9-003',
  'Pclass': '2',
  'Sex': 'male',
  'SibSp': '2',
  'Survived': '0',
  'Ticket': '9993'}

_doc4 = {
  'Age': '12',
  'Cabin': 'ZZ9ZZA',
  'Embarked': 'S',
  'Fare': '42.00',
  'Name': 'Adams, Master Ford',
  'Parch': '2',
  'PassengerId': 'ZZ9-004',
  'Pclass': '2',
  'Sex': 'male',
  'SibSp': '2',
  'Survived': '0',
  'Ticket': '9994'}

_doc5 = {
  'Age': '2',
  'Cabin': 'ZZ9ZZA',
  'Embarked': 'S',
  'Fare': '42.00',
  'Name': 'Adams, Master Marvin',
  'Parch': '2',
  'PassengerId': 'ZZ9-005',
  'Pclass': '2',
  'Sex': 'male',
  'SibSp': '2',
  'Survived': '1',
  'Ticket': '9995'}


# Insert the documents into the collection
_result_ = titanic_col.insert_one(_doc1)
_result_ = titanic_col.insert_one(_doc2)
_result_ = titanic_col.insert_one(_doc3)
_result_ = titanic_col.insert_one(_doc4)
_result_ = titanic_col.insert_one(_doc5)

# Task 3.3 - MongoDB Queries


## Task 3.3.1 
> Write code to count the number of passengers who survived or did not survive, grouped by passenger class (Pclass).

Firstly, I retrieved the basic statistics on survival/death from the databas for reference.

To answer the task, I used an aggregation pipeline with the following stages.

**Pipeline Stages:**
1. Group by Passenger class and Survived fields
2. The 'Project' stage formats the output. The 'cond' operation converts 0 to 'No' and 1 to 'Yes'
3. Sort the results by Passenger class and Survived status.

To print the results table, I use the function defined above 'print_results_table'.

In [69]:
# Basic Stats for Survival - for reference
survived = titanic_col.count_documents({ "Survived": "1" })
died = titanic_col.count_documents({ "Survived": "0" })

print (f"Passengers who survived: {survived}")
print (f"Passengers who did not survive: {died}")
print (f"Total passengers: {survived + died}\n")


result = titanic_col.aggregate(
 [
    {
        '$group': {
            '_id': {
                'Pclass': '$Pclass', 
                'Survived': '$Survived'
            }, 
            'count': {
                '$sum': 1
            }
        }
    }, {
        '$project': {
            '_id': 0, 
            'Pclass': '$_id.Pclass', 
            'Survived': {
                '$cond': {
                    'if': {
                        '$eq': [
                            '$_id.Survived', '1'
                        ]
                    }, 
                    'then': 'Yes', 
                    'else': 'No'
                }
            }, 
            'Count': '$count'
        }
    }, {
        '$sort': {
            'Pclass': 1, 
            'Survived': -1
        }
    }
]
)

# Print the result
print_results_table(result.to_list(), table_title = "Grouped by Passenger class and survival with count in each group")

Passengers who survived: 344
Passengers who did not survive: 552
Total passengers: 896


Grouped by Passenger class and survival with count in each group
--------------------------------------------------
          Pclass        Survived           Count
--------------------------------------------------
               1             Yes             136
               1              No              80
               2             Yes              89
               2              No             100
               3             Yes             119
               3              No             372
--------------------------------------------------
Total Results:  6
--------------------------------------------------



## Task 3.3.2 
> Write code to find the average age of survivors and non-survivors, grouped by passenger class (Pclass). 

The dataset provided the age value as a string, so a pipeline stage was required to convert this to a floating point number.
Initially, this failed in some instances because not all records include a valid age, so another prior pipeline stage was added to discount all records where there is no age recorded.

**Pipeline Stages:**
1. Remove documents with an empty Age field
2. Convert Age to a number field
3. Group by Passenger class and Survived fields
4. The 'Project' stage formats the output. The 'cond' operation converts 0 to 'No' and 1 to 'Yes'. The average age is rounded to an integer.
5. Sort the results by Passenger class and Survived status.

In [70]:
result = titanic_col.aggregate(
[
    {
        '$match': {
            'Age': {
                '$ne': ''
            }
        }
    }, {
        '$addFields': {
            'AgeAsNum': {
                '$toDouble': '$Age'
            }
        }
    }, {
        '$group': {
            '_id': {
                'Pclass': '$Pclass', 
                'Survived': '$Survived'
            }, 
            'AverageAge': {
                '$avg': '$AgeAsNum'
            }
        }
    }, {
        '$project': {
            '_id': 0, 
            'Pclass': '$_id.Pclass', 
            'Survived': {
                '$cond': {
                    'if': {
                        '$eq': [
                            '$_id.Survived', '1'
                        ]
                    }, 
                    'then': 'Yes', 
                    'else': 'No'
                }
            }, 
            'AverageAge': {
                '$round': [
                    '$AverageAge', 0
                ]
            }
        }
    }, {
        '$sort': {
            'Pclass': 1, 
            'Survived': -1
        }
    }
]
)

# Print the result
print_results_table(result.to_list(), table_title = "Grouped by Passenger class and survival, with average age")


Grouped by Passenger class and survival, with average age
--------------------------------------------------
          Pclass        Survived      AverageAge
--------------------------------------------------
               1             Yes            35.0
               1              No            44.0
               2             Yes            26.0
               2              No            33.0
               3             Yes            21.0
               3              No            27.0
--------------------------------------------------
Total Results:  6
--------------------------------------------------



## Task 3.3.3 
Write a code to find the total fare paid by survivors and non-survivors, grouped by gender (Sex).

As above, the fare was stored as a string, so a numeric version was required. Again, potentially empty values of the fare were removed prior to conversion.

**Pipeline Stages:**
1. Remove documents with an empty Fare field
2. Convert Fare to a number field
3. Group by Sex and Survived fields, collect the total fare per group.
4. The 'Project' stage formats the output. The 'cond' operation converts 0 to 'No' and 1 to 'Yes'. The FareTotal field is rounded to 2 decimal places.
5. Sort the results by Sex and Survived status.

In [71]:
result = titanic_col.aggregate(
[
    {
        '$match': {
            'Fare': {
                '$ne': ''
            }
        }
    }, {
        '$addFields': {
            'FareNum': {
                '$toDouble': '$Fare'
            }
        }
    }, {
        '$group': {
            '_id': {
                'Sex': '$Sex', 
                'Survived': '$Survived'
            }, 
            'FareTotal': {
                '$sum': '$FareNum'
            }
        }
    }, {
        '$project': {
            '_id': 0, 
            'Sex': '$_id.Sex', 
            'Survived': {
                '$cond': {
                    'if': {
                        '$eq': [
                            '$_id.Survived', '1'
                        ]
                    }, 
                    'then': 'Yes', 
                    'else': 'No'
                }
            }, 
            'TotalFare': {
                '$round': [
                    '$FareTotal', 2
                ]
            }
        }
    }, {
        '$sort': {
            'Sex': 1, 
            'Survived': -1
        }
    }
]
)

# Print the result
print_results_table(result.to_list(), table_title = "Grouped by Sex and survival, with total fare")



Grouped by Sex and survival, with total fare
--------------------------------------------------
             Sex        Survived       TotalFare
--------------------------------------------------
          female             Yes        12101.69
          female              No         1906.98
            male             Yes         4533.54
            male              No        10361.74
--------------------------------------------------
Total Results:  4
--------------------------------------------------




## Task 3.3.4
Write a code to find the percentage of survivors in each class (Pclass).

I learned that it is possible to create multiple 'project' stages to calculate further based on fields that have been generated previously. The second stage can do some calculations and also must simply pass on the rest down the pipeline; if this is not done, these variables will be lost and not returned with the rest of the documents.

**Pipeline Stages:**
1. Group by Pclass and Survived fields, but only when survived is "True" (1).
2. The first 'project' stage creates a SurvivedPercentage variable from the Survived and PclassTotal variables collected by the group stage.
3. The second project stage creates a DiedPercentage from the previous variables.
4. The third project stage converts the float values to a string and concatenates a '%'. 
5. Sort the results by Pclass.


In [72]:
result = titanic_col.aggregate(
[
    {
        '$group': {
            '_id': '$Pclass', 
            'PClassTotal': {
                '$sum': 1
            }, 
            'Survived': {
                '$sum': {
                    '$cond': [
                        {
                            '$eq': [
                                '$Survived', '1'
                            ]
                        }, 1, 0
                    ]
                }
            }
        }
    }, {
        '$project': {
            '_id': 0, 
            'Pclass': '$_id', 
            'SurvivedPercentage': {
                '$round': [
                    {
                        '$multiply': [
                            {
                                '$divide': [
                                    '$Survived', '$PClassTotal'
                                ]
                            }, 100
                        ]
                    }, 2
                ]
            }
        }
    }, {
        '$project': {
            '_id': 1, 
            'Pclass': '$Pclass', 
            'SurvivedPercentage': '$SurvivedPercentage', 
            'DiedPercentage': {
                '$subtract': [
                    100, '$SurvivedPercentage'
                ]
            }
        }
    }, {
        '$project': {
            '_id': 1, 
            'Pclass': '$Pclass', 
            'SurvivedPercentage': {
                '$concat': [ { '$toString': '$SurvivedPercentage' }, '%' ]
            },
            'DiedPercentage': {
                '$concat': [ { '$toString': '$DiedPercentage' }, '%' ]
            }
        }
    }, {
        '$sort': {
            'Pclass': 1
        }
    }
]
)

# Print the result
print_results_table(result.to_list(), table_title = "Grouped by Passenger Class showing survival and desceased percentage", padding=20)




Grouped by Passenger Class showing survival and desceased percentage
--------------------------------------------------------------
              Pclass  SurvivedPercentage      DiedPercentage
--------------------------------------------------------------
                   1              62.96%              37.04%
                   2              47.09%              52.91%
                   3              24.24%              75.76%
--------------------------------------------------------------
Total Results:  3
--------------------------------------------------------------





## Task 3.3.5 
> Write code to find the passengers who paid the highest fare, along with their details.

Having explored the data, my interpretation of this was to print a list of the top twenty fare paying passengers. There are a number of passengers who paid the very top fare, and a variety of fares below this.

In [73]:
result = titanic_col.aggregate(
[
    {
        '$addFields': {
            'Fare': {
                '$toDouble': '$Fare'
            }
        }
    }, {
        '$sort': {
            'Fare': -1
        }
    }, {
        '$limit': 20
    }
]
)

result = result.to_list()

# Remove unwanted fields
to_remove = ["_id", "Ticket", "Cabin", "Embarked", "PassengerId", "Parch", "SibSp"]
for r in to_remove:
    for row in result:
        if r in row:
            del row[r]

# Print the result
print_results_table(result, table_title = "Top 20 Passengers, sorted by Fare, highest first", padding=25)



Top 20 Passengers, sorted by Fare, highest first
--------------------------------------------------------------------------------------------------------------------------------------------------------
                 Survived                   Pclass                     Name                      Sex                      Age                     Fare
--------------------------------------------------------------------------------------------------------------------------------------------------------
                        1                        1Cardeza, Mr. Thomas Drake                     male                       36                 512.3292
                        1                        1   Lesurer, Mr. Gustave J                     male                       35                 512.3292
                        1                        1         Ward, Miss. Anna                   female                       35                 512.3292
                        0               


## Task 3.3.6 
> Write code to find the number of missing values for each field (e.g., Age, Embarked, etc.).

The following pipeline uses two stages to seatch for empty fields:

**Pipeline Stages:**
1. 'Project' stage to build variables for "Fieldname_Empty" based on whether the field value is equal to "".
2. 'Group' stage to perform a count on these variables.

In [74]:
result = titanic_col.aggregate(
[
    {
        '$project': {
            '_id': 0, 
            'Age_Empty': {
                '$cond': [
                    {
                        '$eq': [
                            '$Age', ''
                        ]
                    }, 1, 0
                ]
            }, 
            'Cabin_Empty': {
                '$cond': [
                    {
                        '$eq': [
                            '$Cabin', ''
                        ]
                    }, 1, 0
                ]
            }, 
            'Embarked_Empty': {
                '$cond': [
                    {
                        '$eq': [
                            '$Embarked', ''
                        ]
                    }, 1, 0
                ]
            }, 
            'Fare_Empty': {
                '$cond': [
                    {
                        '$eq': [
                            '$Fare', ''
                        ]
                    }, 1, 0
                ]
            }, 
            'Name_Empty': {
                '$cond': [
                    {
                        '$eq': [
                            '$Name', ''
                        ]
                    }, 1, 0
                ]
            }, 
            'Parch_Empty': {
                '$cond': [
                    {
                        '$eq': [
                            '$Parch', ''
                        ]
                    }, 1, 0
                ]
            }, 
            'PassengerId_Empty': {
                '$cond': [
                    {
                        '$eq': [
                            '$PassengerId', ''
                        ]
                    }, 1, 0
                ]
            }, 
            'Pclass_Empty': {
                '$cond': [
                    {
                        '$eq': [
                            '$Pclass', ''
                        ]
                    }, 1, 0
                ]
            }, 
            'Sex_Empty': {
                '$cond': [
                    {
                        '$eq': [
                            '$Sex', ''
                        ]
                    }, 1, 0
                ]
            }, 
            'SibSp_Empty': {
                '$cond': [
                    {
                        '$eq': [
                            '$SibSp', ''
                        ]
                    }, 1, 0
                ]
            }, 
            'Survived_Empty': {
                '$cond': [
                    {
                        '$eq': [
                            '$Survived', ''
                        ]
                    }, 1, 0
                ]
            }, 
            'Ticket_Empty': {
                '$cond': [
                    {
                        '$eq': [
                            '$Ticket', ''
                        ]
                    }, 1, 0
                ]
            }
        }
    }, {
        '$group': {
            '_id': 1, 
            'Age': {
                '$sum': '$Age_Empty'
            }, 
            'Cabin': {
                '$sum': '$Cabin_Empty'
            }, 
            'Embarked': {
                '$sum': '$Embarked_Empty'
            }, 
            'Fare': {
                '$sum': '$Fare_Empty'
            }, 
            'Name': {
                '$sum': '$Name_Empty'
            }, 
            'Parch': {
                '$sum': '$Parch_Empty'
            }, 
            'PassengerId': {
                '$sum': '$PassengerId_Empty'
            }, 
            'Pclass': {
                '$sum': '$Pclass_Empty'
            }, 
            'Sex': {
                '$sum': '$Sex_Empty'
            }, 
            'SibSp': {
                '$sum': '$SibSp_Empty'
            }, 
            'Survived': {
                '$sum': '$Survived_Empty'
            }, 
            'Ticket': {
                '$sum': '$Ticket_Empty'
            }
        }
    }
]
)

print("Count of empty fields in the dataset")
# There should only be one 'row':
for row in result:
    for field in field_names:
        print(f"{Fore.CYAN}{field}:{Fore.RESET} {row[field]}")


Count of empty fields in the dataset
Age: 177
Cabin: 687
Embarked: 2
Fare: 0
Name: 0
Parch: 0
PassengerId: 0
Pclass: 0
Sex: 0
SibSp: 0
Survived: 0
Ticket: 0


## Task 3.3.7 
> Write a code to find passengers who survived and have missing Age information. 

This is very straightforward and can be achieved using an aggregation pipeline.

In [75]:
result = titanic_col.aggregate(
[
    {
        '$match': {
            'Survived': '1', 
            'Age': ''
        }
    }
]
)

result = result.to_list()

# Remove unwanted fields
to_remove = ["_id", "Ticket", "Cabin", "Embarked", "PassengerId", "Parch", "SibSp", "Age"]
for r in to_remove:
    for row in result:
        if r in row:
            del row[r]

# Print the result
print_results_table(result, table_title = "Passengers who survived with empty Age field:", padding=25)




Passengers who survived with empty Age field:
-------------------------------------------------------------------------------------------------------------------------------
                 Survived                   Pclass                     Name                      Sex                     Fare
-------------------------------------------------------------------------------------------------------------------------------
                        1                        2Williams, Mr. Charles Eug                     male                       13
                        1                        3  Masselmani, Mrs. Fatima                   female                    7.225
                        1                        3O'Dwyer, Miss. Ellen "Nel                   female                   7.8792
                        1                        1Spencer, Mrs. William Aug                   female                 146.5208
                        1                        3 Glynn, Miss. Mar

## Task 3.3.8 
> Write a code to find the number of passengers who survived, grouped by age group (e.g., 0-10, 11-20, etc.).

This was achieved by using a number of pipeline stages, including a 'switch' stage - which allowed me to create buckets of age groups.

**Pipeline Stages:**
1. Match survivors only and remove records with an empty Age field.
2. Convert the Age field to a floating point number. This is because some of them are recorded as, for example, 26.5 years.
3. Convert the floating point number to an integer. (I found this to fail without converting to a double first.)
4. Generate the AgeGroup buckets using a switch statement. Note that the ages groups are 0-10, 11-20, 21-30 ... 71-80 and 81+. There is also a default value assigned which should never occur in practice. 
5. A group stage is used to count the members of each bucket.
6. A project stage is used to rename the variables for neater presentation.
7. Then finally, the output is sorted by AgeGroup.


In [76]:
result = titanic_col.aggregate(
[
    {
        '$match': {
            'Survived': '1', 
            'Age': {
                '$ne': ''
            }
        }
    }, 
    {
        '$addFields': {
            'AgeDouble': {
                '$toDouble': '$Age'
            }
        }
    }, 
    {
        '$project': {
            'AgeNum': {
                '$toInt': '$AgeDouble'
            }
        }
    }, 
    {
        '$project': {
            'AgeGroup': {
                '$switch': {
                    'branches': [
                        {
                            'case': {
                                '$lte': [
                                    '$AgeNum', 10
                                ]
                            }, 
                            'then': '0-10'
                        }, {
                            'case': {
                                '$and': [
                                    {
                                        '$gt': [
                                            '$AgeNum', 10
                                        ]
                                    }, {
                                        '$lte': [
                                            '$AgeNum', 20
                                        ]
                                    }
                                ]
                            }, 
                            'then': '11-20'
                        }, {
                            'case': {
                                '$and': [
                                    {
                                        '$gt': [
                                            '$AgeNum', 20
                                        ]
                                    }, {
                                        '$lte': [
                                            '$AgeNum', 30
                                        ]
                                    }
                                ]
                            }, 
                            'then': '21-30'
                        }, {
                            'case': {
                                '$and': [
                                    {
                                        '$gt': [
                                            '$AgeNum', 30
                                        ]
                                    }, {
                                        '$lte': [
                                            '$AgeNum', 40
                                        ]
                                    }
                                ]
                            }, 
                            'then': '31-40'
                        }, {
                            'case': {
                                '$and': [
                                    {
                                        '$gt': [
                                            '$AgeNum', 40
                                        ]
                                    }, {
                                        '$lte': [
                                            '$AgeNum', 50
                                        ]
                                    }
                                ]
                            }, 
                            'then': '41-50'
                        }, {
                            'case': {
                                '$and': [
                                    {
                                        '$gt': [
                                            '$AgeNum', 50
                                        ]
                                    }, {
                                        '$lte': [
                                            '$AgeNum', 60
                                        ]
                                    }
                                ]
                            }, 
                            'then': '51-60'
                        }, {
                            'case': {
                                '$and': [
                                    {
                                        '$gt': [
                                            '$AgeNum', 60
                                        ]
                                    }, {
                                        '$lte': [
                                            '$AgeNum', 70
                                        ]
                                    }
                                ]
                            }, 
                            'then': '61-70'
                        }, {
                            'case': {
                                '$and': [
                                    {
                                        '$gt': [
                                            '$AgeNum', 70
                                        ]
                                    }, {
                                        '$lte': [
                                            '$AgeNum', 80
                                        ]
                                    }
                                ]
                            }, 
                            'then': '71-80'
                        }, {
                            'case': {
                                '$gt': [
                                    '$AgeNum', 80
                                ]
                            }, 
                            'then': '81+'
                        }
                    ], 
                    'default': 'Unknown - Something went wrong...!'
                }
            }
        }
    }, 
    {
        '$group': {
            '_id': '$AgeGroup', 
            'count': {
                '$sum': 1
            }
        }
    }, 
    {
        '$project': {
            '_id': 0, 
            'AgeGroup': '$_id', 
            'Count': '$count'
        }
    }, 
    {
        '$sort': {
            'AgeGroup': 1
        }
    }
]    
)


# Print the result
print_results_table(result.to_list(), table_title = "Grouped by Age Group", padding=25)



Grouped by Age Group
----------------------------------------------------
                 AgeGroup                    Count
----------------------------------------------------
                     0-10                       39
                    11-20                       44
                    21-30                       84
                    31-40                       70
                    41-50                       33
                    51-60                       17
                    61-70                        4
                    71-80                        1
----------------------------------------------------
Total Results:  8
----------------------------------------------------




## Task 3.3.9 
> Find the number of passengers who survived, grouped by the first letter of their name.

The format of the names is as follows:

Braund, Mr. Owen Harris


**Pipeline stages:**
1. Match survivors only
2. Extract the first letter of the name using the substrCP operation. (Starting at index 0, taking one character.)
3. A project stage is used to convert these letters to upper case (to avoid names like de'Ath being counted differently.)
4. A group stage is used to count the names starting with each extracted letter.
5. A project stage is used to rename the variables for neater presentation - (otherwise, we get an _id variable).
6. Then the output is sorted by FirstLetter.

Note: It is assumed that 'name' referes to the surname, which is fairly simple to extract as detailed above. To collect the first letter of the forename would require a more complex pipeline. I would use a split on ",", then a second split on "space", and take the first letter of the second resulting token.

In [77]:
result = titanic_col.aggregate(
[
    {
        '$match': {
            'Survived': '1'
        }
    }, {
        '$project': {
            '_id': 0, 
            'FirstLetter': {
                '$substrCP': [
                    '$Name', 0, 1
                ]
            }
        }
    }, {
        '$project': {
            '_id': 1, 
            'FirstLetter': {
                '$toUpper': '$FirstLetter'
            }
        }
    },{
        '$group': {
            '_id': '$FirstLetter', 
            'Count': {
                '$sum': 1
            }
        }
    }, {
        '$project': {
            '_id': 'null', 
            'FirstLetter': '$_id', 
            'Count': 1
        }
    }, {
        '$sort': {
            'FirstLetter': 1
        }
    }
]
)

result = result.to_list()

to_remove = ["_id"]

for r in to_remove:
    for row in result:
        if r in row:
            del row[r]

# Print the result
print_results_table(result, table_title = "Survivors Grouped by First Letter of Name", padding=25)



Survivors Grouped by First Letter of Name
----------------------------------------------------
                    Count              FirstLetter
----------------------------------------------------
                       21                        A
                       35                        B
                       27                        C
                       22                        D
                        3                        E
                       12                        F
                       10                        G
                       32                        H
                        2                        I
                       11                        J
                        9                        K
                       18                        L
                       33                        M
                       15                        N
                        6                        O
                       13          


## Task 3.3.10 
> Write a code to find the median fare paid by passengers in each class (Pclass).

**Pipeline stages:**
1. Match only records with a recorded fare.
2. Convert the fare string to a number.
3. A group stage to groupby Passenger Class, and record the median fare for each.
4. A project stage to rename the variables and produce neater output.
5. Sort by the passenger class for output.

In [78]:
result = titanic_col.aggregate(
[
    {
        '$match': {
            'Fare': {
                '$ne': ''
            }
        }
    }, {
        '$addFields': {
            'FareNum': {
                '$toDouble': '$Fare'
            }
        }
    }, {
        '$group': {
            '_id': {
                'PClass': '$Pclass'
            }, 
            'FareMedian': {
                '$median': {
                    'input': '$FareNum', 
                    'method': 'approximate'
                }
            }
        }
    }, {
        '$project': {
            '_id': 0, 
            'PClass': '$_id.PClass', 
            'MedianFare': '$FareMedian'
        }
    }, {
        '$sort': {
            'PClass': 1
        }
    }
]
)

# Print the result
print_results_table(result.to_list(), table_title = "Median fare by passenger class", padding=25)




Median fare by passenger class
----------------------------------------------------
                   PClass               MedianFare
----------------------------------------------------
                        1                     59.4
                        2                     15.0
                        3                     8.05
----------------------------------------------------
Total Results:  3
----------------------------------------------------



## Task 3.3.11 
> Write a code to find the passengers who were in the same family (same surname). 

The format of the names is as follows:

Braund, Mr. Owen Harris

**Pipeline stages:**
1. The add fields stage is used to create a Surname object, using a split operation to take the surname (the first token).
2. A group operation stage with a push operation is used to collect the Name and Age into an object by Surname. (It is possible that there might be many 'Smith's or 'Jones's, but I am grouping them together as there is no way to tell on Surname alone!)
3. The next stage filters the output so that the results only contain actual families with more than one person. 
4. The 'project' stage makes the output nicer by renaming the '_id' field to Surname.
5. Finally, the Surnames are sorted alphabetically.


In [80]:
result = titanic_col.aggregate(
[
    {
        '$addFields': {
            'Surname': {
                '$arrayElemAt': [
                    { '$split': [ '$Name', ',' ] }, 0
                ]
            }
        }
    }, {
        '$group': {
            '_id': '$Surname', 
            'family': {
                '$push': {
                    'Name': '$Name', 
                    'Age': '$Age'
                }
            }, 
            'count': {
                '$sum': 1
            }
        }
    }, {
        '$match': {
            'count': {
                '$gt': 1
            }
        }
    }, {
        '$project': {
            '_id': 0, 
            'Surname': '$_id', 
            'Family': '$family'
        }
    }, {
        '$sort': {
            'Surname': 1
        }
    }
]
)

# Print the result
for row in result:
    print (f"{Fore.YELLOW}Family Name: {row['Surname']}{Fore.RESET}")
    for family_member in row['Family']:
        print (f"\t{Fore.CYAN}Name:{Fore.RESET} {family_member['Name']}, {Fore.CYAN}Age:{Fore.RESET} {family_member['Age']}")
    print ("")


Family Name: Abbott
	Name: Abbott, Mrs. Stanton (Rosa Hunt), Age: 35
	Name: Abbott, Mr. Rossmore Edward, Age: 16

Family Name: Abelson
	Name: Abelson, Mr. Samuel, Age: 30
	Name: Abelson, Mrs. Samuel (Hannah Wizosky), Age: 28

Family Name: Adams
	Name: Adams, Mr. John, Age: 26
	Name: Adams, Mr. Arthur, Age: 40
	Name: Adams, Mrs. Trillian, Age: 33
	Name: Adams, Master Zaphod, Age: 14
	Name: Adams, Master Ford, Age: 12
	Name: Adams, Master Marvin, Age: 2

Family Name: Ali
	Name: Ali, Mr. Ahmed, Age: 24
	Name: Ali, Mr. William, Age: 25

Family Name: Allen
	Name: Allen, Mr. William Henry, Age: 35
	Name: Allen, Miss. Elisabeth Walton, Age: 29

Family Name: Allison
	Name: Allison, Miss. Helen Loraine, Age: 2
	Name: Allison, Master. Hudson Trevor, Age: 0.92
	Name: Allison, Mrs. Hudson J C (Bessie Waldo Daniels), Age: 25

Family Name: Andersson
	Name: Andersson, Mr. Anders Johan, Age: 39
	Name: Andersson, Miss. Erna Alexandra, Age: 17
	Name: Andersson, Miss. Ellis Anna Maria, Age: 2
	Name: Ande

## Task 3.3.12 
> Write a code to find the survival rate for male and female passengers. 

This pipeline is similar to the code for Task 3.3.4, except the grouping is by Sex.

**Pipeline Stages:**
1. Group by Sex and Survived fields, but only when survived is "True" (1). 
2. The first 'project' stage creates a SurvivalRate variable from the Survived and Sex variables collected by the group stage.
3. The second project stage appends a 'percentage' symbol. The float variable must be converted to a string first before the % can be concatenated.

In [81]:
result = titanic_col.aggregate(
[
    {
        '$group': {
            '_id': '$Sex', 
            'Sex': {
                '$sum': 1
            }, 
            'Survived': {
                '$sum': {
                    '$cond': [
                        {
                            '$eq': [
                                '$Survived', '1'
                            ]
                        }, 1, 0
                    ]
                }
            }
        }
    }, {
        '$project': {
            '_id': 0, 
            'Sex': '$_id', 
            'SurvivalRate': {
                '$round': [
                    {
                        '$multiply': [
                            {
                                '$divide': [
                                    '$Survived', '$Sex'
                                ]
                            }, 100
                        ]
                    }, 2
                ]
            }
        }
    }, {
        '$project': {
            '_id': 1, 
            'Sex': '$Sex', 
            'SurvivalRate': {
                '$concat': [ { '$toString': '$SurvivalRate' }, '%' ]
            }
        }
    }
]
)

# Print the result
print_results_table(result.to_list(), table_title = "Survival Rate by Sex", padding=8)




Survival Rate by Sex
------------------
     SexSurvival
------------------
    male   19.1%
  female  73.97%
------------------
Total Results:  2
------------------



## Close the Connection

In [82]:
client.close()